In [ ]:
import scvi
import scanpy as sc
import pandas as pd

In [ ]:
adata = sc.read_h5ad("./Data/RNA_ATAC/BMMC/BMMC.h5ad")

adata_gex = adata[:, adata.var['modality'] == "Gene Expression"]
adata_atac = adata[:, adata.var['modality'] == "Peaks"]
print(adata_gex)
print(adata_atac)

In [ ]:
# We can now use the organizing method from scvi to concatenate these anndata
adata_mvi = scvi.data.organize_multiome_anndatas(adata)
adata_mvi = adata_mvi[:, adata_mvi.var["modality"].argsort()].copy()
sc.pp.filter_genes(adata_mvi, min_cells=int(adata_mvi.shape[0] * 0.01))

In [ ]:
scvi.model.MULTIVI.setup_anndata(adata_mvi, batch_key="batch")

model = scvi.model.MULTIVI(
    adata_mvi,
    n_genes=(adata_mvi.var["modality"] == "Gene Expression").sum(),
    n_regions=(adata_mvi.var["modality"] == "Peaks").sum(),
)
# model.view_anndata_setup()

In [ ]:
model.train()

In [ ]:
MULTIVI_LATENT_KEY = "X_multivi"

adata_mvi.obsm[MULTIVI_LATENT_KEY] = model.get_latent_representation()
adata_mvi.write("MultiVI_BMMC.h5ad")